# Phase shifter of a silicon modulator

A silicon traveling-wave modulator changes the phase of the light with a PN junction in a rib waveguide: applying a voltage moves the edge of the depletion region, and the free carriers that come and go change the refractive index and the absorption of the guided mode. This guide simulates that junction and compares it with the modulator of Baehr-Jones *et al.*, *Optics Express* **20**(11), 12014 (2012).

It is simulated with GDSFactory TCAD, based on [DEVSIM](https://devsim.org) (drift-diffusion for the junction) and femwell (finite elements for the optical mode), with a closed-form transmission-line model for the loaded electrode.

> **Result**
>
> - **Junction capacitance.** The simulated C(V) is within 2.3 % of the paper's Fig. 2d at all ten biases (rms 1.2 %). The doping profile has one fitted number (the lateral straggle of the implants, 0.20 µm, least-squares fit, interval 0.196 to 0.224 µm), so this shows that the fit worked, not that C(V) is predicted: see the details.
> - **Modulation efficiency.** From -0.1 V up, the simulated V$_\pi$L is within 10.0 % of the paper's mid-arm value. This is a prediction that follows from the fitted profile.
> - **Group index** 3.90 against the paper's 3.86.
> - **Implant loss** at 0 V is **5.3 dB/cm** against the paper's estimate of **10 dB/cm** (and its own prediction of **6.7 dB/cm**). That is outside both: a limit of the one-parameter profile.
> - **Loaded electrode** at 10 GHz: 37.6 ohm, RF index 3.65 and **10.3 dB/cm** of RF loss, against 37.0 ohm, 3.75 and **10.8 dB/cm** in the paper; the electro-optic 3 dB bandwidth is 10.9 GHz against about 10 GHz.

The long version, with the doping fit, the sensitivity scans, the loss checks and the electro-optic response, is [Phase shifter of a silicon modulator: simulation details](tcad_bj2012_modulator_details.md).

## The device

The cross-section follows the paper's Fig. 1b: a 220 nm silicon rib, 0.5 µm wide, on a 90 nm slab, with the PN junction in the middle of the rib. The heavily doped contact regions start 0.95 µm from the rib edge and reach the 3.55 µm mark, where the aluminum lands on them. The paper does not state the contact doping (1e20 cm$^{-3}$ is assumed) or the junction position (the rib centre is assumed); it quotes peak doping of 7e17 (p) and 5e17 cm$^{-3}$ (n) and the sheet resistance of the slab (10 and 7 k$\Omega$/sq), which sets the slab doping.

The paper's implants are segmented along the light's direction (10 µm period, 1 µm gaps), so only `FILL` = 90 % of the length carries a junction. The simulation is a cross-section of a segment, while the measured quantities are averages over the whole length: every per-length result is multiplied by `FILL` before it meets the paper.

Reference values (read by eye from the paper's figures, about one marker of uncertainty) are in `data/bj2012_reference.json`; every number this guide quotes is in `data/rf_benchmark/bj2012_modulator.json`.

In [ ]:
import json
import logging
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np

from gsim.common.cross_section import build_doped_cross_section
from gsim.common.stack.extractor import Layer
from gsim.common.stack.materials import MATERIALS_DB, make_doped_materials

# The mesher's advice on air boxes targets 3D driven simulations; it does not
# apply to these cross-section solves.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

REF_PATH = next(
    p
    for p in (
        Path("data/bj2012_reference.json"),
        Path("nbs/data/bj2012_reference.json"),
    )
    if p.exists()
)
REF = json.loads(REF_PATH.read_text(encoding="utf-8"))
from IPython.display import Markdown, display

STORED = json.loads(
    (REF_PATH.parent / "rf_benchmark" / "bj2012_modulator.json").read_text(
        encoding="utf-8"
    )
)
ROWS = {row["id"]: row for row in STORED["rows"]}

In [ ]:
# Lengths in um, from the reference file.
WAVELENGTH_UM = REF["geometry"]["wavelength_um"]
CENTER_Y = -20.0  # junction position along y (um)
RIB_HEIGHT = REF["geometry"]["rib_height_um"]
SLAB_HEIGHT = REF["geometry"]["slab_height_um"]
RIB_HALF = REF["geometry"]["rib_width_um"] / 2
SLAB_WIDTH = REF["geometry"]["contact_start_from_rib_edge_um"]
CONTACT_WIDTH = (
    REF["geometry"]["rib_edge_to_via_side_um"]
    - REF["geometry"]["contact_start_from_rib_edge_um"]
)
METAL_THICKNESS = REF["geometry"]["lower_metal_thickness_um"]
# Fraction of the device length that carries a junction (Fig. 1a inset).
FILL = (
    1
    - REF["geometry"]["implant_striation_gap_um"]
    / REF["geometry"]["implant_striation_period_um"]
)

# Regions from the n-side contact to the p-side one:
# name: (dopant, width um, height um)
REGIONS = {
    "n_contact": ("donor", CONTACT_WIDTH, SLAB_HEIGHT),
    "n_slab": ("donor", SLAB_WIDTH, SLAB_HEIGHT),
    "n_rib": ("donor", RIB_HALF, RIB_HEIGHT),
    "p_rib": ("acceptor", RIB_HALF, RIB_HEIGHT),
    "p_slab": ("acceptor", SLAB_WIDTH, SLAB_HEIGHT),
    "p_contact": ("acceptor", CONTACT_WIDTH, SLAB_HEIGHT),
}
ELECTRODES = {"n_contact": "cathode_metal", "p_contact": "anode_metal"}

gf.gpdk.PDK.activate()
component = gf.Component()
layers, spans = {}, {}
y = CENTER_Y - sum(width for _, width, _ in REGIONS.values()) / 2
for i, (name, (_, width, height)) in enumerate(REGIONS.items()):
    spans[name] = (y, y + width)
    rect = component << gf.c.rectangle((10.0, width), layer=(30, i))
    rect.y = y + width / 2
    layers[name] = Layer(
        name=name,
        gds_layer=(30, i),
        zmin=0.0,
        zmax=height,
        thickness=height,
        material=name,
        layer_type="dielectric",
        mesh_resolution="medium" if "contact" in name else "fine",
    )
    y += width
for i, (region, metal) in enumerate(ELECTRODES.items()):
    y0, y1 = spans[region]
    rect = component << gf.c.rectangle((10.0, y1 - y0), layer=(46, i))
    rect.y = (y0 + y1) / 2
    layers[metal] = Layer(
        name=metal,
        gds_layer=(46, i),
        zmin=SLAB_HEIGHT,
        zmax=SLAB_HEIGHT + METAL_THICKNESS,
        thickness=METAL_THICKNESS,
        material="aluminum",
        layer_type="conductor",
        mesh_resolution="fine",
    )

materials = make_doped_materials([(name, 0.0) for name in REGIONS], permittivity=11.9)
sellmeier = next(
    m for m in MATERIALS_DB["silicon"].dispersion_models if m.type == "sellmeier"
)
for properties in materials.values():
    properties.dispersion_models.append(sellmeier)
stack, _ = build_doped_cross_section(
    component,
    axis="x",
    value=0.0,
    substrate_thickness=2.0,
    doping={"layer_specs": layers, "materials": materials},
    verbose=False,
)

DOPED_SPAN = (spans["n_contact"][0], spans["p_contact"][1])
RIB_SPAN = (spans["n_rib"][0], spans["p_rib"][1])
CHARGE_WINDOW = (DOPED_SPAN[0] - 0.5, DOPED_SPAN[1] + 0.5)
OPTICAL_WINDOW = (CENTER_Y - 2.0, CENTER_Y + 2.0)
OPTICAL_WINDOW_Z = (-1.0, RIB_HEIGHT + 1.0)
JUNCTION_BOX = (*RIB_SPAN, 0.0, RIB_HEIGHT, 0.01)  # (y0, y1, z0, z1, size) um

for name, (dopant, width, height) in REGIONS.items():
    y0, y1 = spans[name]
    print(
        f"{name:10s} y = {y0:7.2f} .. {y1:7.2f} um, {width:4.2f} um wide, "
        f"{height * 1e3:3.0f} nm tall, {dopant}"
    )

In [ ]:
import meshio
from matplotlib.collections import PolyCollection

REGION_COLORS = {}  # one colour per region name, stable across figures


def plot_cross_section(mesh_path, *, ax, title=None, xlim=None, ylim=None):
    """Draw a cross-section mesh with one colour per region."""
    mesh = meshio.read(mesh_path, file_format="gmsh")
    points = mesh.points[:, :2]
    names = {tag: name for name, (tag, dim) in mesh.field_data.items() if dim == 2}
    triangles = mesh.cells_dict["triangle"]
    tags = mesh.cell_data_dict["gmsh:physical"]["triangle"]
    handles = []
    for tag in sorted(set(tags), key=names.get):
        name = names[tag]
        color = REGION_COLORS.setdefault(
            name, plt.get_cmap("tab20")(len(REGION_COLORS) % 20)
        )
        ax.add_collection(
            PolyCollection(
                points[triangles[tags == tag]],
                rasterized=True,  # thousands of triangles: keep the figure small
                facecolors=color,
                edgecolors="k",
                linewidths=0.1,
            )
        )
        handles.append(plt.Rectangle((0, 0), 1, 1, color=color, label=name))
    ax.autoscale_view()
    ax.set_aspect("equal")
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set(xlabel="y (um)", ylabel="z (um)", title=title)
    ax.legend(
        handles=handles,
        fontsize=7,
        ncol=min(len(handles), 6),
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
    )
    return ax


from itertools import pairwise

from gsim import tcad


def build_charge_sim(profiles, *, output_dir, refined_um=0.02, junction_box_um=0.01):
    """A meshed `tcad.Simulation` of the junction with the given doping profiles."""
    sim = tcad.Simulation()
    sim.geometry(component, stack=stack)
    sim.domain(
        plane="x=0",
        window=CHARGE_WINDOW,
        margin_x=2.0,
        margin_y=2.0,
        z_above=1.5,
        z_below=1.0,
    )
    sim.materials(insulators={"sio2": stack.materials["sio2"]["permittivity"]})
    for profile in profiles:
        sim.doping.add(profile)
    sim.contacts.add(
        name="cathode", semiconductor="n_contact", electrode="cathode_metal"
    )
    sim.contacts.add(name="anode", semiconductor="p_contact", electrode="anode_metal")
    for left, right in pairwise(REGIONS):
        junction = (left, right) == ("n_rib", "p_rib")
        sim.interfaces.add(
            name="junction" if junction else f"{left}_{right}",
            region_a=left,
            region_b=right,
        )
    sim.solver(
        mesh_preset="coarse",
        refined_mesh_size_um=refined_um,
        max_mesh_size_um=40.0,
        refinement_boxes=[(*RIB_SPAN, 0.0, RIB_HEIGHT, junction_box_um)],
    )
    mesh = sim.mesh(output_dir=output_dir)
    return sim, mesh


def uniform_profiles(concentration_cm3):
    """One uniform StepDoping per region from a {region: cm^-3} dict."""
    return [
        tcad.StepDoping(
            region=name, dopant_type=REGIONS[name][0], concentration_cm3=value
        )
        for name, value in concentration_cm3.items()
    ]


# The paper's quoted peak doping, and the slab doping that gives its sheet resistance.
P_PEAK, N_PEAK = REF["doping"]["p_peak_cm3"], REF["doping"]["n_peak_cm3"]
from scipy.optimize import brentq

from gsim.common.carriers import MobilityModel

Q_C = 1.602176634e-19
MOBILITY = MobilityModel.masetti_silicon()


def sheet_resistance_ohm_sq(concentration_cm3, mobility_cm2, thickness_um):
    """Sheet resistance of a uniformly doped, fully ionised film."""
    return 1.0 / (
        Q_C * mobility_cm2(concentration_cm3) * concentration_cm3 * thickness_um * 1e-4
    )


SHEET_P = REF["doping"]["slab_sheet_resistance_p_kohm_per_sq"] * 1e3
SHEET_N = REF["doping"]["slab_sheet_resistance_n_kohm_per_sq"] * 1e3
SLAB_P_CM3 = brentq(
    lambda c: sheet_resistance_ohm_sq(c, MOBILITY.holes_cm2, SLAB_HEIGHT) - SHEET_P,
    1e16,
    1e19,
)
SLAB_N_CM3 = brentq(
    lambda c: sheet_resistance_ohm_sq(c, MOBILITY.electrons_cm2, SLAB_HEIGHT) - SHEET_N,
    1e16,
    1e19,
)
print(
    f"p slab: {SLAB_P_CM3:.2e} cm^-3 -> {sheet_resistance_ohm_sq(SLAB_P_CM3, MOBILITY.holes_cm2, SLAB_HEIGHT) / 1e3:.2f} kohm/sq"
    f" (mu_p = {float(MOBILITY.holes_cm2(SLAB_P_CM3)):.0f} cm^2/Vs)"
)
print(
    f"n slab: {SLAB_N_CM3:.2e} cm^-3 -> {sheet_resistance_ohm_sq(SLAB_N_CM3, MOBILITY.electrons_cm2, SLAB_HEIGHT) / 1e3:.2f} kohm/sq"
    f" (mu_n = {float(MOBILITY.electrons_cm2(SLAB_N_CM3)):.0f} cm^2/Vs)"
)


from scipy.special import erfc

SIGMA_UM = 0.20  # calibrated lateral straggle (um): fitted to Fig. 2d, not a measured value


def graded_profiles(sigma_um):
    """Erfc-graded n and p implants across the rib; slab and contact doping as above."""
    junction_y = CENTER_Y
    plateau = {  # region: (donor plateau, acceptor plateau) in cm^-3
        "n_rib": (N_PEAK, P_PEAK),
        "p_rib": (N_PEAK, P_PEAK),
        "n_slab": (SLAB_N_CM3, SLAB_P_CM3),
        "p_slab": (SLAB_N_CM3, SLAB_P_CM3),
    }
    profiles = []
    for region, (n_plateau, p_plateau) in plateau.items():
        profiles.append(
            tcad.CallableDoping(
                region=region,
                dopant_type="donor",
                function=lambda y, _z, nd=n_plateau: (
                    nd * 0.5 * erfc((y - junction_y) / (np.sqrt(2) * sigma_um))
                ),
            )
        )
        profiles.append(
            tcad.CallableDoping(
                region=region,
                dopant_type="acceptor",
                function=lambda y, _z, na=p_plateau: (
                    na * 0.5 * erfc(-(y - junction_y) / (np.sqrt(2) * sigma_um))
                ),
            )
        )
    profiles.extend(
        tcad.StepDoping(
            region=region, dopant_type=REGIONS[region][0], concentration_cm3=1e20
        )
        for region in ("n_contact", "p_contact")
    )
    return profiles

In [ ]:
graded_sim, graded_mesh = build_charge_sim(
    graded_profiles(SIGMA_UM), output_dir="./tcad-bj2012/graded"
)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), height_ratios=(1, 2))
plot_cross_section(
    graded_mesh.mesh_path,
    ax=axes[0],
    title="The simulated cross-section: slab, rib and the two contacts",
    ylim=(-0.3, 1.2),
)
axes[0].get_legend().remove()  # the close-up below carries it
plot_cross_section(
    graded_mesh.mesh_path,
    ax=axes[1],
    title="Around the junction: 10 nm elements across the rib",
    xlim=(CENTER_Y - 0.6, CENTER_Y + 0.6),
    ylim=(-0.05, 0.3),
)
plt.tight_layout()
plt.show()

In [ ]:
# One bias point, 0 V: the doping and the carriers DEVSIM finds.
F_PER_CM_TO_FF_PER_UM = 1e11  # 1 F/cm of depth = 1e15 fF / 1e4 um
point = graded_sim.sweep(np.array([0.0]), contact="cathode").points[0]
carriers = point.carriers


def signed_log(x):
    """Log10 of |x| with the sign kept (net doping: n side positive)."""
    return np.sign(x) * np.log10(np.maximum(np.abs(x), 1.0))


fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
fields = (
    ("Net doping (signed log10 cm^-3)", signed_log(carriers.net_doping_cm3), "RdBu_r"),
    (
        "Electrons (log10 cm^-3)",
        np.log10(np.maximum(carriers.electrons_cm3, 1.0)),
        "viridis",
    ),
    ("Holes (log10 cm^-3)", np.log10(np.maximum(carriers.holes_cm3, 1.0)), "viridis"),
)
for ax, (title, values, cmap) in zip(axes, fields, strict=True):
    tri = ax.tricontourf(
        carriers.x_um, carriers.y_um, values, levels=30, cmap=cmap, rasterized=True
    )
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("y (um)")
    ax.set_xlim(CENTER_Y - 1.5, CENTER_Y + 1.5)
    ax.set_ylim(0.0, 0.3)
    fig.colorbar(tri, ax=ax)
axes[0].set_ylabel("z (um)")
plt.show()
print(
    f"0 V: C = {FILL * point.capacitance_f_per_cm * F_PER_CM_TO_FF_PER_UM:.3f} fF/um "
    "per device length"
)

In [ ]:
from gsim.common.carrier_transfer import transfer_carriers
from gsim.common.carriers import (
    PlasmaDispersionModel,
    carrier_absorption_cm,
    carrier_index_shift,
    permittivity_perturbation,
)
from gsim.common.modes import select_line_mode
from gsim.femwell import boundary_field_ratio, epsilon_by_region, solve_modes
from gsim.palace import BoundaryModeSim

DISPERSION_MODELS = {
    "Soref-Bennett": PlasmaDispersionModel.soref_1550(),
    "Nedeljkovic": PlasmaDispersionModel.nedeljkovic_1550(),
}


def cross_section_mesh(
    component, stack, *, plane, output_dir, window=None, window_z=None, boxes=None
):
    """Mesh a cross-section of a component in oxide and return the mesh path."""
    sim = BoundaryModeSim()
    sim.set_output_dir(output_dir)
    sim.set_stack(stack)
    sim.set_geometry(component)
    sim.set_airbox(
        margin_x=2.0, margin_y=2.0, z_above=1.5, z_below=1.0, material="sio2"
    )
    sim.set_cross_section(plane, window=window, window_z=window_z)
    sim.mesh(
        preset="coarse",
        refined_mesh_size=0.05,
        max_mesh_size=40.0,
        refinement_boxes=boxes,
        verbose=False,
    )
    return sim.mesh_path


optical_mesh = cross_section_mesh(
    component,
    stack,
    plane="x=0",
    output_dir="./tcad-bj2012/optical",
    window=OPTICAL_WINDOW,
    window_z=OPTICAL_WINDOW_Z,
    boxes=[JUNCTION_BOX],
)


def optical_mode(carriers, dispersion, wavelength_um=WAVELENGTH_UM):
    """Guided mode with one carrier map on the optical mesh.

    Returns the femwell mode, the carriers on the optical mesh, the absorption
    (cm^-1) and the complex permittivity per element.
    """
    base = epsilon_by_region(optical_mesh, stack, wavelength_um=wavelength_um)
    doped = [name for name in REGIONS if name in base]
    on_mesh = transfer_carriers(
        carriers, optical_mesh, at="elements", fill=0.0, regions=doped
    )
    index_shift = carrier_index_shift(
        on_mesh.electrons_cm3, on_mesh.holes_cm3, model=dispersion
    )
    absorption_cm = carrier_absorption_cm(
        on_mesh.electrons_cm3, on_mesh.holes_cm3, model=dispersion
    )
    epsilon = np.array([base[name] for name in on_mesh.region], dtype=complex)
    for i, name in enumerate(on_mesh.region):
        if name in doped:
            epsilon[i] = permittivity_perturbation(
                n0=np.sqrt(base[name].real),
                dn=index_shift[i],
                dalpha_cm=absorption_cm[i],
                wavelength_um=wavelength_um,
            )
    modes = solve_modes(
        optical_mesh,
        epsilon=epsilon,
        wavelength_um=wavelength_um,
        metallic_boundaries=True,
    )
    mode = select_line_mode(modes)
    assert boundary_field_ratio(mode) < 0.01, "the optical window clips the mode"
    return mode, on_mesh, absorption_cm, epsilon


def solve_optical_mode(carriers, dispersion, wavelength_um=WAVELENGTH_UM):
    """Complex n_eff of the guided mode with one carrier map on the optical mesh."""
    return complex(optical_mode(carriers, dispersion, wavelength_um)[0].n_eff)


# |Im n_eff| is an amplitude decay: 4 pi |Im n_eff| / lambda per unit length in power.
LOSS_DB_PER_CM_PER_IMAG = 40 * np.pi / np.log(10) / (WAVELENGTH_UM * 1e-4)

In [ ]:
primary = "Soref-Bennett"
mode, on_mesh, absorption_cm, _epsilon = optical_mode(
    point.carriers, DISPERSION_MODELS[primary]
)
n_complex = complex(mode.n_eff)
loss_0v = FILL * LOSS_DB_PER_CM_PER_IMAG * abs(n_complex.imag)  # device average
print(f"n_eff = {n_complex.real:.5f}, implant loss at 0 V = {loss_0v:.2f} dB/cm")
stored = ROWS["loss-sb-0v-mzi"]["value"]
assert abs(loss_0v / stored - 1) <= 1e-2, (loss_0v, stored)
print(f"matches the stored row loss-sb-0v-mzi ({stored:.3f} dB/cm, rel. 1e-2)")

fig, ax = plt.subplots(figsize=(8, 3.6))
mode.plot_component("E", "x", part="abs", boundaries=False, colorbar=False, ax=ax)
for artist in ax.collections:  # dense field mesh: raster, not vector
    artist.set_rasterized(True)
ax.set(
    xlim=(CENTER_Y - 1.2, CENTER_Y + 1.2),
    ylim=(-0.4, 0.7),
    xlabel="y (um)",
    ylabel="z (um)",
    title="Intensity of the guided mode at 1.55 um (femwell), white: region edges",
)
ax.set_aspect("equal")
plt.show()

## How it is simulated

1. **Junction (DEVSIM).** Poisson and drift-diffusion on the doped cross-section at every bias give the electron and hole maps, and a small-signal AC solve gives the capacitance and the series resistance of the junction. The doping profile is the paper's quoted peaks and slab doping with the abrupt junction replaced by error-function tails of width $\sigma$: the one fitted number.
2. **Carriers to light (femwell).** Each carrier map becomes an index change and an absorption through the Soref-Bennett fit (the Nedeljkovic fit is the comparison). femwell solves the guided mode on the same cross-section: its complex effective index gives the phase shift per volt, V$_\pi$L and the implant loss.
3. **Loaded electrode (closed form).** The junction's series-RC branch is added to the shunt admittance of the metal-only electrode, per unit length. The loaded line's impedance, RF index and loss follow, and the electro-optic response from averaging the RF voltage along the co-propagating light.

Everything runs locally and nothing is submitted to a cloud service. The [details notebook](tcad_bj2012_modulator_details.md) holds the doping fit, the sensitivity scans, the checks of the loss and the electro-optic response.

In [ ]:
CV_BIASES = REF["capacitance"]["bias_v_reverse"]  # V on the cathode = reverse bias
VPIL_BIASES = REF["vpil_V_cm"]["bias_v_reverse"]
cv = [ROWS[f"cv-{k:02d}"] for k in range(len(CV_BIASES))]
vp = [ROWS[f"vpil-{k:02d}"] for k in range(len(VPIL_BIASES))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.plot(
    [r["bias_v"] for r in cv],
    [r["reference_value"] for r in cv],
    "ks",
    label="paper, Fig. 2d (read by eye)",
)
ax.plot(
    [r["bias_v"] for r in cv],
    [r["value"] for r in cv],
    "o-",
    color="C3",
    label="DEVSIM, graded junction (sigma fitted)",
)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C per device length (fF/um)")
ax.set_title("Junction capacitance: a fit")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
volts = [r["bias_v"] for r in vp]
ax.plot(volts, REF["vpil_V_cm"]["top_arm"], "ks", label="paper, top arm")
ax.plot(volts, REF["vpil_V_cm"]["bottom_arm"], "k^", label="paper, bottom arm")
ax.plot(volts, [r["value"] for r in vp], "o-", color="C3", label="simulation")
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("V_pi L (V cm)")
ax.set_title("Modulation efficiency: a prediction")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def complex_of(row, key="value"):
    """The (complex) value or reference of a stored row."""
    return complex(row[key], row.get(f"{key}_imag") or 0.0)


def agreement(row):
    """Relative error of a row against its reference, or None."""
    if row.get("reference_value") is None:
        return None
    reference = complex_of(row, "reference_value")
    return abs(complex_of(row) - reference) / abs(reference)


SHOWN = (
    "cv-02",
    "vpil-04",
    "n-group",
    "loss-sb-0v-mzi",
    "loss-sb-0v-pred",
    "ll-tcad-z0",
    "ll-tcad-n",
    "ll-tcad-alpha",
    "eo-f3db-input",
)
lines = [
    "| Quantity | Simulation | Paper | Difference | Role | Solver, order, mesh, hardware |",
    "|---|---|---|---|---|---|",
]
for key in SHOWN:
    row = ROWS[key]
    error = agreement(row)
    where = f"at {row['bias_v']:g} V " if row.get("bias_v") is not None else ""
    lines.append(
        f"| {row['quantity']} {where}({row['unit']}) "
        f"| {row['value']:.4g} "
        f"| {row['reference_value']:.4g} {row.get('reference_label') or ''} "
        f"| {f'{error:+.1%}' if error is not None else '-'} "
        f"| {row['role']} "
        f"| {row['solver']} {row['solver_version']}; order {row['element_order']}; "
        f"{row['mesh']}; {row['hardware']} |"
    )
display(Markdown("\n".join(lines)))

## What agrees and what does not

- **The capacitance is a fit.** The paper does not give the doping profile, and its quoted peak doping as a uniform abrupt profile gives 1.4 to 2.0 times the measured capacitance. One number, the lateral straggle $\sigma$, brings C(V) within 8.0 % of Fig. 2d at all ten biases. Agreement there only shows that the fit worked. The same number gave 3.1 % before the DEVSIM assembly was corrected (gsim #480, which lowers the junction capacitance; with it C settles as the mesh is refined, where the old assembly did not); it was not re-fitted, and the details notebook shows that 0.22 µm would bring the curve back within 3 %.
- **V$_\pi$L and the group index are predictions** that inherit the fitted profile: V$_\pi$L within 10.0 % of the mid-arm value from -0.1 V up, the group index within 1 % of 3.86. The most forward point (-0.4 V) is not scored: it is the end of the digitised curve, where the two arms differ by 15 % and a small error in the bias is a large one in V$_\pi$L.
- **The implant loss does not agree.** 5.3 dB/cm against the paper's estimate of 10 dB/cm and its own 6.7 dB/cm. The one-parameter profile cannot fit both the capacitance and the loss: C(V) sees the doping at the edges of the depletion region, the loss sees the whole mode. The paper's 10 dB/cm is itself an estimate (the insertion loss of the Mach-Zehnder minus the y-junctions and an unstated waveguide loss), good to about ±1.5 dB/cm.
- **The loaded electrode agrees with the paper** to 1.6 % in impedance, 2.6 % in index and 4.8 % in loss at 10 GHz, but its metal-only line is the paper's own measurement and the junction values come from a TCAD calibrated to it: a consistency check, not a prediction.

<details>
<summary>Estimator choices and the 6 dB rule</summary>

V$_\pi$L is the small-signal value of the paper's Fig. 2c, so it is evaluated from the local slope of $\Delta n_{eff}(V)$ (a cubic spline through the sweep); read as a chord from 0 V it would fall far short of the paper's 4.60 and 5.35 V cm at 4.5 V, so the paper's data follow the local slope. At -0.4 V four slope estimators agree within 6 %.

The electro-optic bandwidth is evaluated with the paper's convention: the 25 $\Omega$ termination, referenced to the voltage at the line input. The paper states that the 3 dB point is reached at around 6 dB of RF loss at the end of the line; for a velocity-matched, matched line the response $|H| = (1 - e^{-x})/x$ falls to $1/\sqrt 2$ at $x = 0.74$ Np, 6.4 dB. The same 25 $\Omega$ load driven from 50 $\Omega$ and referenced to the source gives a higher figure (19.1 GHz); that is an illustration of the termination, not a quantity the paper reports.

The derivations and the cells are in the [details notebook](tcad_bj2012_modulator_details.md): the doping fit (Step 1), the checks of the loss and of the slope (Step 2), the loaded line, the 6 dB rule and the striations (Step 3).

</details>

## Limits

- The capacitance is fitted, and everything that follows inherits the fit. That Fig. 2d is a device-length average is an assumption: without the striation fill the loss and V$_\pi$L would change by about 10 %.
- The implant loss is outside the paper's, and no profile in the one-parameter family fits both the capacitance and the loss. A two-parameter profile is the next step and is not implemented.
- The loaded-electrode model is one line: the paper's ripple, the probe and the cable are not in it.
- The digitised values carry about one marker of uncertainty: 0.01 fF/µm in the capacitance and 0.1 to 0.2 V cm in V$_\pi$L.

## Next

- [RF electrode of a silicon modulator](palace_bj2012_electrode.md): the metal-only electrode that the junction loads, simulated with Palace and femwell.
- [Phase shifter of a silicon modulator: simulation details](tcad_bj2012_modulator_details.md): the long version.